# 12 · DreamBooth-LoRA baseline on Stable Diffusion 1.5 (Phase 8)

**T4 GPU runtime.** One LoRA per image (rank 8, lr 1e-4, **600 steps**, batch 1, 512 px, fp16, 8-bit Adam,
no prior preservation), for the 8 images in `data/manifests/lora8.csv` (3 natural + 5 OOD).
Then 30 samples per image (30 DDIM steps, CFG 7.5, seeds 10..39), resized back to the input's size for evaluation.

- **Resumable:** a LoRA already on Drive is not retrained; finished samples are skipped.
- **Limitation (documented):** the plan's 2-image dev set for tuning steps/lr is dropped on the free tier;
  these are the defaults of the diffusers example, untuned. No regularisation images.
- Costs per image (train s, sample s/img, peak VRAM) go to `tables/phase8_lora_cost.json`.

In [ ]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK    = "12_lora_train"
STEPS       = 600
SESSION_MIN = 100
N_SAMPLES   = 30
SEED0       = 10

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
DIFFUSERS_VERSION = "0.35.1"
# same pins as notebook 06 (newest peft rejects Colab's torchao; we never use torchao)
!pip uninstall -q -y torchao
!pip install -q diffusers=={DIFFUSERS_VERSION} transformers==4.53.2 accelerate==1.9.0 peft==0.17.1 bitsandbytes pandas
!wget -q -O /content/train_dreambooth_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{DIFFUSERS_VERSION}/examples/dreambooth/train_dreambooth_lora.py
!accelerate config default

## Train one LoRA per image (skips images already trained)

In [ ]:
import time, json, glob, csv, subprocess, torch, pandas as pd
rows = list(csv.DictReader(open(f"{PROJECT}/data/manifests/lora8.csv", encoding="utf-8")))
LORA = f"{DRIVE_ROOT}/checkpoints/lora"
COST = f"{DRIVE_ROOT}/tables/phase8_lora_cost.json"
cost = json.load(open(COST)) if os.path.exists(COST) else {}
def abs_path(r):                                   # manifest paths are relative to the manifest folder
    return os.path.normpath(os.path.join(f"{PROJECT}/data/manifests", r["path"]))
def weights_ready(i):
    return bool(glob.glob(f"{LORA}/{i}/pytorch_lora_weights.*"))

remaining = float(SESSION_MIN) * 60
for r in rows:
    i = r["id"]
    if weights_ready(i):
        print(i, "already trained"); continue
    if remaining < 600:
        print("session budget used up; rerun later"); break
    inst = f"/content/lora_inst/{i}"; os.makedirs(inst, exist_ok=True)
    shutil.copy(abs_path(r), f"{inst}/img.png")
    out = f"/content/lora_out/{i}"
    cmd = (f"accelerate launch /content/train_dreambooth_lora.py --pretrained_model_name_or_path=stable-diffusion-v1-5/stable-diffusion-v1-5 "
           f"--instance_data_dir={inst} --instance_prompt='{r['prompt']}' --output_dir={out} --resolution=512 "
           f"--train_batch_size=1 --gradient_accumulation_steps=1 --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 "
           f"--max_train_steps={STEPS} --rank=8 --mixed_precision=fp16 --gradient_checkpointing --use_8bit_adam "
           f"--checkpointing_steps=100000 --seed=0")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    ret = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0; remaining -= dt
    assert ret == 0 and glob.glob(f"{out}/pytorch_lora_weights.*"), f"LoRA training FAILED for {i} (see the error above)"
    os.makedirs(f"{LORA}/{i}", exist_ok=True)
    for f in glob.glob(f"{out}/pytorch_lora_weights.*"):
        shutil.copy(f, f"{LORA}/{i}/")
    cost.setdefault(i, {})["train_s"] = round(dt, 1)
    json.dump(cost, open(COST, "w"), indent=2)
    print(f"{i}: trained in {dt/60:.1f} min")

## Sample 30 images per trained LoRA into the standard output layout (resized back to the input size)

In [ ]:
from diffusers import StableDiffusionPipeline
from PIL import Image
OUT = f"{DRIVE_ROOT}/outputs/phase8_lora/lora_r8_s{STEPS}__lora"
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/job.json"):
    json.dump({"experiment": "phase8_lora", "job": f"lora_r8_s{STEPS}", "method": "lora_sd15",
               "params": {"rank": 8, "lr": 1e-4, "steps": STEPS, "resolution": 512, "sampler_steps": 30, "cfg": 7.5},
               "images": [{"id": r["id"], "path": abs_path(r)} for r in rows]}, open(f"{OUT}/job.json", "w"), indent=2)
pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16,
                                               safety_checker=None).to("cuda")
pipe.set_progress_bar_config(disable=True)
for r in rows:
    i = r["id"]
    if not weights_ready(i):
        print(i, "no LoRA yet"); continue
    d = f"{OUT}/{i}"; os.makedirs(d, exist_ok=True)
    todo = [s for s in range(SEED0, SEED0 + N_SAMPLES) if not os.path.exists(f"{d}/seed_{s:04d}.png")]
    if not todo:
        continue
    W, H = Image.open(abs_path(r)).size
    pipe.load_lora_weights(f"{LORA}/{i}")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    for s in todo:
        im = pipe(r["prompt"], num_inference_steps=30, guidance_scale=7.5,
                  generator=torch.Generator("cuda").manual_seed(s)).images[0]
        im.resize((W, H), Image.LANCZOS).save(f"{d}/seed_{s:04d}.png")
    cost.setdefault(i, {}).update(sample_s_per_img=round((time.time() - t0) / len(todo), 2),
                                  sample_peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 2))
    json.dump(cost, open(COST, "w"), indent=2)
    pipe.unload_lora_weights()
    print(i, len(todo), "samples", cost[i])
pd.DataFrame(cost).T

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")